# Milestone2 — definitive Colab GPU worker
Floating memories • 1080p24 • 12 samples • OptiX ray tracing + GPU OpenImageDenoise • ProRes 422 + 16-bit depth. Select a GPU runtime and Run all. Authorize the Google account with write access to the linked job folder. Drive folder IDs avoid ambiguous mounted paths. Run one worker per job.
Rerun any interrupted cell, or Run all: verified files are reused, local worker locking prevents overlapping launches in one runtime, and conflicting remote files are rejected. Do not run the same job in two Colab runtimes.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
import os, json, hashlib, shutil, io, time, subprocess, tarfile
creds,_=default()
drive_api=build('drive','v3',credentials=creds,cache_discovery=False)
JOB_ID='1qYdX-8ia0THTt0KFI5qzKBjt7xNJjA13'
JOB_JSON_ID='1fztia838tPSsw3c-_j4zpkmc2-sBr2Wq'
INPUT_ID='14Bq_F6WTdkIAb6DLd7ljcR72fj-wdFwV'
folder=drive_api.files().get(fileId=JOB_ID,fields='id,name,capabilities(canAddChildren)',supportsAllDrives=True).execute()
assert folder['capabilities']['canAddChildren'], 'Authorize an account with write access to the job folder'
print('Verified Drive job:',folder['name'],folder['id'])
def children(parent):
    q=f"'{parent}' in parents and trashed=false"
    token=None; result=[]
    while True:
        r=drive_api.files().list(q=q,fields='nextPageToken,files(id,name,size,md5Checksum,mimeType)',pageSize=1000,pageToken=token,supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        result.extend(r.get('files',[]))
        token=r.get('nextPageToken')
        if not token:return result
def lookup(parent,name):
    matches=[x for x in children(parent) if x['name']==name]
    assert len(matches)<=1, f'Duplicate Drive name: {name}'
    return matches[0] if matches else None
def folder_id(parent,name):
    f=lookup(parent,name)
    if f:
        assert f['mimeType']=='application/vnd.google-apps.folder'
        return f['id']
    return drive_api.files().create(body={'name':name,'parents':[parent],'mimeType':'application/vnd.google-apps.folder'},fields='id',supportsAllDrives=True).execute()['id']
def file_hash(path,algorithm='md5'):
    with path.open('rb') as f:return hashlib.file_digest(f,algorithm).hexdigest()
def index_files(parent):
    result={}
    for f in children(parent):
        if f['name'] in result:raise RuntimeError('Duplicate Drive name: '+f['name'])
        result[f['name']]=f
    return result
def download(fid,dst,expected=None):
    expected=expected or drive_api.files().get(fileId=fid,fields='id,name,size,md5Checksum',supportsAllDrives=True).execute()
    def matches(path):
        return path.is_file() and path.stat().st_size==int(expected['size']) and file_hash(path)==expected['md5Checksum']
    if matches(dst):return False
    dst.parent.mkdir(parents=True,exist_ok=True)
    tmp=dst.with_suffix(dst.suffix+'.downloading')
    with tmp.open('wb') as f:
        loader=MediaIoBaseDownload(f,drive_api.files().get_media(fileId=fid,supportsAllDrives=True),chunksize=32*1024*1024)
        done=False
        while not done:
            status,done=loader.next_chunk(num_retries=5)
    if not matches(tmp):raise RuntimeError("Downloaded checksum mismatch: "+dst.name)
    os.replace(tmp,dst)
    return True
def upload(src,parent,existing=None):
    with src.open('rb') as f: md5=hashlib.file_digest(f,'md5').hexdigest()
    if existing and existing.get('md5Checksum')==md5 and int(existing.get('size',-1))==src.stat().st_size:return existing
    # Refuse conflicting existing content instead of overwriting committed media.
    if existing:raise RuntimeError('Conflicting Drive file: '+src.name)
    media=MediaFileUpload(str(src),resumable=True,chunksize=8*1024*1024)
    args=dict(media_body=media,fields='id,name,size,md5Checksum',supportsAllDrives=True)
    if existing:req=drive_api.files().update(fileId=existing['id'],**args)
    else:req=drive_api.files().create(body={'name':src.name,'parents':[parent]},**args)
    result=None
    while result is None: _,result=req.next_chunk(num_retries=5)
    assert result['md5Checksum']==md5 and int(result['size'])==src.stat().st_size, 'Upload verification failed'
    return result
cfg=json.loads(drive_api.files().get_media(fileId=JOB_JSON_ID,supportsAllDrives=True).execute())
assert cfg['version']==2 and cfg['scene']=='floating' and cfg['count']==2420
print('Job:',cfg['scene'],cfg['count'],'frames')


In [ ]:
import pathlib, subprocess, urllib.request, hashlib, tarfile, json, shutil
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
base='https://download.blender.org/release/Blender5.0/'
name='blender-5.0.1-linux-x64.tar.xz'
archive=Path('/content')/name
exe=Path('/content/blender-5.0.1-linux-x64/blender')
if not exe.is_file():
    urllib.request.urlretrieve(base+name,archive)
    expected=next(line.split()[0] for line in urllib.request.urlopen(base+'blender-5.0.1.sha256').read().decode().splitlines() if line.endswith(name))
    with archive.open('rb') as f: assert hashlib.file_digest(f,'sha256').hexdigest()==expected, 'Blender checksum mismatch'
    subprocess.run(['tar','-xJf',str(archive),'-C','/content'],check=True)

import shutil
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'FFmpeg required'


In [ ]:
work=Path('/content/milestone2-worker')/cfg['sha256'][:16]
work.mkdir(parents=True,exist_ok=True)
bundle=work/'input.tar'
def digest(path):
    with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
if not bundle.exists() or digest(bundle)!=cfg['sha256']:
    print('Downloading verified geometry package (2.32 GB)',flush=True)
    download(INPUT_ID,bundle)
assert digest(bundle)==cfg['sha256'], 'Input checksum mismatch'
# Refuse extraction/render changes while a worker in this runtime is active.
import fcntl
from contextlib import contextmanager
@contextmanager
def worker_lock():
    with (work/'worker.lock').open('a') as lock:
        try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError('A worker is already running in this runtime')
        try:yield
        finally:fcntl.flock(lock,fcntl.LOCK_UN)
def patch_renderer(source):
    old='scene.cycles.denoiser="OPTIX" if a.device=="OPTIX" else "OPENIMAGEDENOISE"'
    new='scene.cycles.denoiser="OPENIMAGEDENOISE"\n    scene.cycles.denoising_use_gpu=True'
    if new in source and old not in source:return source
    if source.count(old)!=1:raise RuntimeError('Unexpected renderer revision')
    return source.replace(old,new,1)
def prepare_cache():
    # Reuse an extracted cache only when all expected archive members exist.
    stamp=work/'extracted.json'
    with tarfile.open(bundle) as t:
        members=t.getmembers()
        ready=stamp.is_file() and stamp.read_text()==cfg['sha256']
        ready=ready and all((work/m.name).is_file() and ((work/m.name).stat().st_size==m.size or m.name=='blender/render_cache.py') for m in members if m.isfile())
        if not ready:
            stamp.unlink(missing_ok=True)
            t.extractall(work,filter='data')
            tmp=stamp.with_suffix('.tmp');tmp.write_text(cfg['sha256']);os.replace(tmp,stamp)
    renderer=work/'blender/render_cache.py'
    source=renderer.read_text()
    patched=patch_renderer(source)
    if patched!=source:
        tmp=renderer.with_suffix('.tmp');tmp.write_text(patched);os.replace(tmp,renderer)
    print('Verified cache prepared; OptiX + GPU OIDN',flush=True)
with worker_lock():prepare_cache()

meta=json.loads((work/'cache/manifest.json').read_text())
assert meta['count']==cfg['count'] and meta['settings']['buildHash']==cfg['buildHash']
out=work/'render.prores';out.mkdir(exist_ok=True)
remote=folder_id(JOB_ID,'render.prores')
chunks_id=folder_id(remote,'chunks')
depth_id=folder_id(remote,'depth')
signature=out/'job-settings.json'
signature.write_text(json.dumps(cfg,sort_keys=True))
old=lookup(remote,signature.name)
if old:
    assert json.loads(drive_api.files().get_media(fileId=old['id'],supportsAllDrives=True).execute())==cfg,'Changed job settings'
else:upload(signature,remote)
remote_chunks=index_files(chunks_id)
remote_depth=index_files(depth_id)
with worker_lock():
    for name,mark in sorted(remote_chunks.items()):
        if not name.endswith('.json'):continue
        dst=out/'chunks'/name
        download(mark['id'],dst,mark)
        d=json.loads(dst.read_text())
        mov=dst.with_suffix('.mov')
        download(remote_chunks[mov.name]['id'],mov,remote_chunks[mov.name])
        for i in range(d['start'],d['start']+d['count']):
            p=out/'depth'/f'frame-{i:06d}.png'
            download(remote_depth[p.name]['id'],p,remote_depth[p.name])
print('Verified cache and restored committed chunks',flush=True)


In [ ]:
# Upload each verified chunk, with its completion marker last.
def sync_chunks():
    global remote_chunks, remote_depth
    marks=sorted((out/'chunks').glob('chunk-*.json'))
    if not any(m.name not in remote_chunks for m in marks):return
    remote_chunks=index_files(chunks_id)
    remote_depth=index_files(depth_id)
    for mark in marks:
        if mark.name in remote_chunks:
            assert file_hash(mark)==remote_chunks[mark.name]['md5Checksum'], 'Conflicting committed chunk'
            continue
        d=json.loads(mark.read_text())
        for i in range(d['start'],d['start']+d['count']):
            p=out/'depth'/f'frame-{i:06d}.png'
            remote_depth[p.name]=upload(p,depth_id,remote_depth.get(p.name))
        mov=mark.with_suffix('.mov')
        remote_chunks[mov.name]=upload(mov,chunks_id,remote_chunks.get(mov.name))
        remote_chunks[mark.name]=upload(mark,chunks_id)
        print('Verified Drive chunk:',d['start'],'frames:',d['count'],flush=True)
cmd=[str(exe),'-b','--factory-startup','--python-exit-code','1','-P',str(work/'blender/render_cache.py'),'--','--cache',str(work/'cache'),'--output',str(out),'--format','PNG','--passes','beauty','--prores','--device','OPTIX','--ffmpeg',shutil.which('ffmpeg'),'--ffprobe',shutil.which('ffprobe')]
for k in ('width','height','samples','bounces','noise','min_samples','diffuse','glossy','chunk_size','depth_far'):
    cmd+=['--'+k.replace('_','-'),str(cfg[k])]
def run_worker():
    global remote_chunks, remote_depth
    remote_chunks=index_files(chunks_id)
    remote_depth=index_files(depth_id)
    # Verify the OptiX compute device before launching; do not silently fall back.
    subprocess.run([str(exe),'-b','--factory-startup','--python-exit-code','1','--python-expr',
        "import bpy; p=bpy.context.preferences.addons['cycles'].preferences; p.compute_device_type='OPTIX'; p.refresh_devices(); assert any(d.type=='OPTIX' for d in p.devices), 'No OptiX GPU detected'"],check=True)
    import threading
    print('Launching OptiX + GPU OIDN at',cfg['samples'],'samples',flush=True)
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    def stream_logs():
        for line in proc.stdout:
            print('BLENDER:',line.rstrip(),flush=True)
    log_thread=threading.Thread(target=stream_logs,daemon=True)
    log_thread.start()
    try:
        while proc.poll() is None:
            sync_chunks()
            time.sleep(5)
    finally:
        if proc.poll() is None:
            proc.terminate()
            proc.wait()
        log_thread.join(timeout=10)
        sync_chunks()
    assert proc.returncode==0, 'Render interrupted; rerun to resume'
    for name in ('depth-metadata.json','media.json'):
        upload(out/name,remote,lookup(remote,name))
    print('Complete: https://drive.google.com/drive/folders/'+remote)

with worker_lock():
    run_worker()
